code for annotate the obs file to get basic metadata summary for portal

In [ ]:
import pandas as pd
import numpy as np
import cellxgene_census
import os, sys, json
from typing import Any
import math

from openai import OpenAI
# Read the key from file
with open("openai_key.txt", "r") as f:
    openai_key = f.read().strip()

# Set environment variable for OpenAI client
os.environ["OPENAI_API_KEY"] = openai_key

## set up

In [ ]:
# input output 
data_dir = "combined_UCE_5neuro_b3a873b2-7981-4088-afb9-b4e8b8e05dca"
metadata_file = os.path.join(data_dir, "obs.tsv.gz") # expect there is a column called dataset_id
metadata_annotated_file = os.path.join(data_dir, "obs_annotated.tsv.gz")
metadata_summary_file = os.path.join(data_dir, "metadata_summary.json")
wrangling_notes_file = os.path.join(data_dir, "wrangling_notes.txt")
system_prompt_file_normalized_age_bucket = os.path.join(data_dir, "system_prompt_normalized_age_bucket.txt")
system_prompt_file_normalized_organism = os.path.join(data_dir, "system_prompt_normalized_organism.txt")

# info files
ucsc_file = "datasetLists/ucsc_datasets.txt"
manual_column_config_file = "datasetLists/manual_column.json"

In [ ]:
fwrangle = open(wrangling_notes_file,'w')

In [ ]:
# usage example 
# json_str = to_json_safe(publication_summary, indent=2)

def to_json_safe(obj: Any, **json_kwargs) -> str:
    """
    Serialize Python objects to valid JSON.
    Converts NaN / ±inf to null and enforces JSON compliance.
    """

    def normalize(o):
        if isinstance(o, dict):
            return {k: normalize(v) for k, v in o.items()}
        if isinstance(o, list):
            return [normalize(v) for v in o]
        if isinstance(o, float):
            if math.isnan(o) or math.isinf(o):
                return None
        return o

    safe_obj = normalize(obj)
    return json.dumps(safe_obj, allow_nan=False, **json_kwargs)


## Load metadata
meta_data

In [ ]:
meta_data = pd.read_csv(metadata_file, sep ='\t')
meta_data.columns

## datasets publication information
**dataset_info** dictionary key is "dataset_id"<br>
values are also dictionary with keys "collection_doi_label", "collection_doi", "dataset_title" are value keys

In [ ]:
# dataset info from cellxgene census                                                                                                                                             
latest_census = cellxgene_census.open_soma(census_version = "latest")
latest_census_datasets = latest_census["census_info"]["datasets"].read().concat().to_pandas()
dataset_info_df = latest_census_datasets[["dataset_id", "collection_doi_label", "collection_doi", "dataset_title"]]
dataset_info_df = dataset_info_df.set_index("dataset_id")
dataset_info = dataset_info_df.to_dict(orient="index")
dataset_info
print("cellxgene datasets:", len(dataset_info))

# dataset info from ucsc datasets
ucsc_df = pd.read_csv(ucsc_file, sep="\t", comment = "#")
ucsc_df = ucsc_df[["dataset_id", "collection_doi_label", "collection_doi", "dataset_title"]]
ucsc_df = ucsc_df.set_index("dataset_id")
ucsc = ucsc_df.to_dict(orient="index")
ucsc
print("ucsc datasets:", len(ucsc))

# combine ucsc with cellxgene 
for idx, vals in ucsc.items():
    if idx in dataset_info:
        print("duplicated datasets detected", idx)
    else:
        dataset_info[idx] = vals
print("all datasets:", len(dataset_info))

## annotate with publication information get raw data information
**add columns: "collection_doi" "collection_doi_label" "dataset_title"**
"publication_summary" for metadata summary json

In [ ]:
ann_columns = ["collection_doi", "collection_doi_label", "dataset_title"]

collection_doi_map = {
    k: v.get("collection_doi")
    for k, v in dataset_info.items()
}

collection_doi_label_map = {
    k: v.get("collection_doi_label")
    for k, v in dataset_info.items()
}

dataset_title_map = {
    k: v.get("dataset_title")
    for k, v in dataset_info.items()
}

ann_column_map ={
    "collection_doi" : collection_doi_map,
    "collection_doi_label" : collection_doi_label_map,
    "dataset_title": dataset_title_map
}
note_map ={
    "collection_doi" : "publication doi",
    "collection_doi_label" : "publication doi label",
    "dataset_title": "dataset title"
}
# annotate with publication information
for column in ann_columns:
    meta_data[column] = meta_data["dataset_id"].map(ann_column_map[column])

def get_Raw_data_summary (meta_data):

    # Count occurrences per dataset_id
    counts = (
        meta_data["dataset_id"]
        .value_counts()
        .reset_index(name="count")
    )

    counts["label"] = counts["dataset_id"].apply(build_url)

    # Convert to list-of-dicts
    result = counts[["label", "count"]].to_dict(orient="records")
    return result

# Build raw data URLs
def build_url(dataset_id):
    if dataset_id.startswith("ucsc"):
        return f"s3://ucsc-h5ad/{dataset_id}.h5ad"
    else:
        return f"{cxg_s3_path}/{dataset_id}.h5ad"

stable_census = cellxgene_census.open_soma(census_version="stable")
cxg_s3_path = stable_census.uri.rsplit("/", 2)[0] + "/h5ads"

# collect summary info
raw_pub_summary = (
    meta_data
    .groupby(["collection_doi_label", "collection_doi", "dataset_id"], dropna=False)
    .size()
    .reset_index(name="count")
)

# add wrangling notes about raw data information
raw_pub_summary["uri"] = raw_pub_summary["dataset_id"].apply(build_url)
uri_list = raw_pub_summary["uri"].astype(str).tolist()
note = "Download raw data from the following locations:"
fwrangle.write(f'{note}\n')
for uri in uri_list:
    fwrangle.write(f'{uri}\n')
    fwrangle.flush()

# publication metadata
publication_summary = []

for (label, doi), group in raw_pub_summary.groupby(
    ["collection_doi_label", "collection_doi"], dropna=False
):
    publication_summary.append({
        "label": label,
        "doi": doi,
        "cell_count": int(group["count"].sum()),
        "raw_data": [
            {
                "uri": row["uri"],
                "cell_count": int(row["count"]),
            }
            for _, row in group.iterrows()
        ],
    })

publication_summary

# add wrangling notes
note = "Combine .obs fields from source h5ad files"
fwrangle.write(f"{note}\n")
fwrangle.flush()

# add wrangling notes for annotate with publication information
for column in ann_columns:
    note = f"Add column '{column}', populate with {note_map[column]} from source data"
    fwrangle.write(f'{note}\n')
    fwrangle.flush()

## custom information in manual_column.json
**custom_map**

In [ ]:
# custum column information such as cell_type or filling in missing columns using static values
with open(manual_column_config_file, "r") as f:
    custom_map = json.load(f)
custom_map

## annotate with custom_map information
**add columns**

exmaple 1

For each row in meta_data:

Default:
custom_cell_type = meta_data["cell_type"]

Override for specific dataset_id:
Use a different column (e.g. "supercluster_term") as specified in celltype_df

example 2

missing organism column

Assign organism Homo sapien

In [ ]:
# For each row in meta_data:

## Default behavior (most datasets):
### Use meta_data["cell_type"]

## Override behavior (some dataset_ids):
### Use different source columns specified in celltype_df
### Example: use "supercluster_term" instead of "cell_type"

mapping = {
    "cell_type": "custom_cell_type",
    "cell_type_ontology_term_id":"custom_cell_type_ontology_term_id"
}
dataset_ids_data = set(meta_data["dataset_id"].unique())

for dataset_id_from_map, cfg in custom_map.items():
    if dataset_id_from_map not in dataset_ids_data:
        continue
    
    mask = meta_data["dataset_id"] == dataset_id_from_map
    
    for metadata_param, param_cfg in cfg.items():
        if not isinstance(param_cfg, dict):
            continue            
        value = param_cfg.get("value")
        type = param_cfg.get("type")
        
        collection_doi_label = dataset_info[dataset_id_from_map]["collection_doi_label"]
        dataset_title = dataset_info[dataset_id_from_map]["dataset_title"]
        
        if type == "static":
            meta_data.loc[mask, metadata_param] = value
            note = (
                f"Set '{metadata_param}' = '{value}' for dataset '{dataset_id_from_map}' "
                f"'{collection_doi_label}' '{dataset_title}'"
            )
            fwrangle.write(f"{note}\n")
            fwrangle.flush()
            print(note)
        elif type == "column":
            new_column = mapping[metadata_param]

            if new_column not in meta_data.columns:
                # 1. Note the Default Mapping
                # (Copying the standard metadata_param to your new custom_column)
                meta_data[new_column] = meta_data[metadata_param]
                default_note = f"Initialized '{new_column}' from '{metadata_param}'"
                fwrangle.write(f"{default_note}\n")
                fwrangle.flush()
                print(default_note)
            
            source_col = value
            meta_data.loc[mask, new_column] = meta_data.loc[mask, source_col]
            # 2. Note the Custom Mapping Override
            custom_note  = ( 
                f"Map column '{source_col}' -> '{new_column}' "
                f"for dataset '{dataset_id_from_map}' '{collection_doi_label}' '{dataset_title}'"
            )
            fwrangle.write(f"{custom_note}\n")
            fwrangle.flush()
            print(custom_note)

## normalized organism information
**add column "normalized_organism"**

In [ ]:
system_prompt = """
You are a biological metadata normalization assistant.

Your task is to normalize heterogeneous organism entries into standardized organism names
and assign the corresponding ontology identifier.

You will be given both:
- the ontology term ID ("organism_ontology_term_id")
- the raw label ("organism")

Instructions:
- Use the ontology term ID as the primary source of truth whenever it is available.
- Use the raw label only if the ontology ID is missing, unknown, or cannot be interpreted.
- Normalize each entry to a single accepted scientific name.
- Use standard binomial nomenclature (Genus species).
- Prefer NCBI Taxonomy conventions.
- Do not make assumptions beyond the information provided.

Your raw label may include:
- common names (e.g. "Human", "mouse", "rat")
- scientific names (e.g. "Homo sapiens", "Mus musculus")
- abbreviations (e.g. "H. sapiens")
- strain or subspecies information (e.g. "C57BL/6 mouse", "human iPSC")
- capitalization or spelling variants
- ambiguous or higher-level taxa (e.g. "mammal", "vertebrate")
- unknown or missing values

---

### STANDARDIZED ORGANISM OUTPUT

- Use the **accepted scientific name** (binomial) whenever possible.
- Capitalize genus, lowercase species (e.g. "Homo sapiens").
- Assign the corresponding **NCBI Taxonomy ID** for the normalized organism.
- If the organism cannot be confidently resolved to a species:
  - Use `"unknown"` for both name and ontology ID.
  - Do NOT guess.

---

### Normalization rules (ordered):

Step 0: Unknown (highest priority)
- If the ontology ID or label indicates "unknown", "unspecified", "unclassified",
  or cannot be resolved → assign:
  - normalized organism: "unknown"
  - normalized ontology ID: "unknown"

Step 1: Ontology-driven resolution
- If a valid organism ontology ID is provided:
  - Resolve it to the corresponding accepted scientific name.
  - Assign the matching NCBI Taxonomy ID.
  - Ignore strain, subspecies, or cell line details.

Step 2: Scientific names in label
- If the label already contains a valid binomial scientific name:
  - Normalize capitalization and spacing only.
  - Assign the corresponding NCBI Taxonomy ID.

Step 3: Common names
- Map common names to their accepted scientific name
  (e.g. "human" → "Homo sapiens").
- Assign the corresponding NCBI Taxonomy ID.

Step 4: Abbreviations
- Expand abbreviations where unambiguous
  (e.g. "H. sapiens" → "Homo sapiens").
- Assign the corresponding NCBI Taxonomy ID.

Step 5: Higher-level or ambiguous taxa
- Labels such as "mammal", "vertebrate", "rodent", or other non-species terms
  → assign:
    - normalized organism: "unknown"
    - normalized ontology ID: "unknown"

---

### Output format (JSON ONLY):

Return a JSON object with this structure:

{
  "annotations": [
    {
      "organism_ontology_term_id": "...",
      "organism": "...",
      "normalized_organism": "...",
      "normalized_organism_ontology_term_id": "..."
    }
  ]
}

Each input entry must appear exactly once.  
Do not include explanations or commentary.
"""

In [ ]:
# Step 1: compute counts per original label/ontology

organism_counts = (
    meta_data
    .groupby(
        ["organism_ontology_term_id", "organism"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .dropna(subset=["organism_ontology_term_id", "organism"], how="all")
)


organism_entries = organism_counts.to_dict(orient="records")

# Step 2: send to OpenAI for normalization

user_message = {
    "development_stage_entries": organism_entries
}

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_message)}
    ]
)

content = response.output_text
annotations = json.loads(content)["annotations"]

# Convert back to DataFrame
normalized_organism = pd.DataFrame(annotations)

# Step 2: merge into meta_data, with new column normalized_age_bucket
meta_data = meta_data.merge(
    normalized_organism[
        [
            "organism",
            "organism_ontology_term_id",
            "normalized_organism"
        ]
    ],
    on=["organism_ontology_term_id", "organism"],
    how="left",
)
note = f"Populate 'normalized_organism' column using OpenAI gpt-4.1 model with 'system_prompt_normalized_organism.txt'"
fwrangle.write(f"{note}\n")
fwrangle.flush()

# QA
df = pd.DataFrame(normalized_organism)

print(
    df[["organism", "organism_ontology_term_id", "normalized_organism"]]
    .to_string(index=False)
)

# Save system prompt to file
with open(system_prompt_file_normalized_organism, "w") as f:
    f.write(system_prompt)

## normalized developmental stage information

"normalized_age_bucket_summary" for summary json
"normalized_age_bucket_ontology_term_id_summary" for summary json

In [ ]:
# human specific

system_prompt = """
You are a biological metadata normalization assistant.

Your task is to normalize heterogeneous "development_stage" entries into standardized age buckets.

This task applies ONLY to human (Homo sapiens) data.
All normalization and ontology mappings must use human developmental stage concepts (HsapDv).

You will be given both:
- the ontology term ID ("development_stage_ontology_term_id")
- the raw label ("development_stage")

Instructions:
- Use the ontology term ID as the primary source of truth whenever it is available.
- Use the raw label only if the ontology ID is missing, unknown, or cannot be interpreted.
- Normalize each entry into a standardized age bucket as defined below.
- Do not make assumptions beyond the information provided.

Your raw label may include:
- exact ages (e.g. "42-year-old human stage")
- age ranges (e.g. "25–44 year-old human stage")
- decades (e.g. "sixth decade stage")
- developmental terms (e.g. "embryonic human stage", "adult stage")
- prenatal timing (e.g. "16th week post-fertilization", "Carnegie stage 20")
- infancy or childhood descriptions
- unknown or ambiguous labels

---

### STANDARDIZED AGE BUCKETS (ordered):

- prenatal_embryonic             (blastula, Carnegie stages, weeks post-fertilization ≤ 8)
- prenatal_fetal                 (weeks post-fertilization > 8 until birth)
- newborn                        (0-1 month)
- infant                         (1-23 months)
- child                          (2-12 years)
- adolescent                     (13-18 years)
- young_adult                    (19-24 years)
- 25-44_year_old                 (25-44 years)
- middle_aged                    (45-64 years)
- older_adult                    (65-79 years)  
- elderly                        (80+ years)
- adult_unspecified              (labels like "adult", "human adult stage")
- postnatal_unspecified          (postnatal but no age resolution)
- unknown                        (explicitly unknown or cannot be inferred)

---

Use HsapDv whenever possible.  
Only use EFO if no suitable HsapDv term exists.

- prenatal_embryonic  
  → HsapDv:0000002 (embryonic stage)

- prenatal_fetal  
  → HsapDv:0000037 (fetal stage)

- newborn  
  → HsapDv:0000082 (newborn stage)

- infant  
  → HsapDv:0000083 (infant stage)

- child  
  → HsapDv:0000081 (child stage)

- adolescent
  → HsapDv:0000086 (adolescent stage)

- young_adult 
  → HsapDv:0000089 (young adult stage)

- 25-44_year_old 
  → HsapDv:0000090 (25-44 year-old human stage)

- middle_aged  
  → HsapDv:0000092 (45-64 year-old human stage)

- older_adult 
  → HsapDv:0000094 (65-79 year-old stage)

- elderly  
  → HsapDv:0000095 (elderly 80+ year-old)

- adult_unspecified  
  → HsapDv:0000087 (adult stage)

- postnatal_unspecified  
  → unknown

- unknown  
  → unknown

---
### Classification rules:

Step 0: Unknown (highest priority)
- If the ontology ID or label indicates "unknown", "unclassified", or cannot be interpreted → assign "unknown".

Step 1: Prenatal stages
- Carnegie stages → prenatal_embryonic
- Blastula, embryonic → prenatal_embryonic
- Post-fertilization weeks:
    - ≤ 8 weeks → prenatal_embryonic
    - > 8 weeks → prenatal_fetal
- LMP month references → prenatal_fetal

Step 2: Newborn / infant
- "newborn", "0–28 days" → newborn
- months old (<24 months) → infant
- "under-2-year-old" → infant

Step 3: Childhood
- 2–12 years → child
- 13-18 years → adolescent

Step 4: Adults (numeric or decade-based)
- 19–24 → young_adult
- 25-44 → 25-44_year_old 
- 45–64 → middle_adult
- 65–79 → older_adult
- ≥ 80 → elderly
- Decade-based terms (e.g. "sixth decade") should be mapped accordingly.

Step 5: Adult but unspecified
- Labels like "adult stage", "human adult stage", "early adulthood", "late adulthood"
  → adult_unspecified

Step 6: Postnatal but unspecified
- "postnatal stage", "infant stage" (without age)
  → postnatal_unspecified

---

### Output format (JSON ONLY):

Return a JSON object with this structure:

{
  "annotations": [
    {
      "development_stage_ontology_term_id": "...",
      "development_stage": "...",
      "normalized_age_bucket": "..."
      "normalized_age_bucket_ontology_term_id": "..."
    }
  ]
}

Each input entry must appear exactly once.  
Do not include explanations or commentary.

"""

In [ ]:
# Step 1:
development_stage_counts = (
    meta_data
    .groupby(
        ["development_stage_ontology_term_id", "development_stage"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

development_stage_entries = development_stage_counts.to_dict(orient="records")

user_message = {
    "development_stage_entries": development_stage_entries
}

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1", # "gpt-4o"
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_message)}
    ]
)

note = f"Populate 'normalized_age_bucket' using OpenAI gpt-4.1 model with 'system_prompt_normalized_age_bucket.txt'"
fwrangle.write(f"{note}\n")
fwrangle.flush()

# Extract JSON from the response
content = response.output_text
annotations = json.loads(content)["annotations"]

# Convert back to DataFrame
normalized_age_bucket = pd.DataFrame(annotations)
#normalized_age_bucket

In [ ]:
# Step 2: normalized_age_bucket_summary
# Create a mapping from development_stage -> normalized_age_bucket
stage_to_bucket = dict(
    zip(
        normalized_age_bucket["development_stage"],
        normalized_age_bucket["normalized_age_bucket"]
    )
)

# Map normalized_age_bucket for summary (without modifying meta_data)
mapped_buckets = meta_data["development_stage"].map(stage_to_bucket)

# Count occurrences
counts = mapped_buckets.value_counts().rename_axis("label").reset_index(name="count")

# Apply canonical ordering
AGE_BUCKET_ORDER = [
    "prenatal_embryonic",
    "prenatal_fetal",
    "newborn",
    "infant",
    "child",
    "adolescent",
    "young_adult",
    "25-44_year_old",
    "middle_aged",
    "older_adult",
    "elderly",
    "adult_unspecified",
    "postnatal_unspecified",
    "unknown",
]

counts["label"] = pd.Categorical(counts["label"], categories=AGE_BUCKET_ORDER, ordered=True)
counts = counts.sort_values("label")
    
# Convert to list-of-dicts for JSON summary
normalized_age_bucket_summary = (
    counts
    .dropna(subset=["label"])
    .rename(columns={"count": "cell_count"})
    .to_dict(orient="records")
)

normalized_age_bucket_summary

In [ ]:
# Step 3: normalized_age_bucket_ontology_term_id_summary
# Create a mapping from development_stage -> normalized_age_bucket_ontology_id
stage_to_bucket = dict(
    zip(
        normalized_age_bucket["development_stage"],
        normalized_age_bucket["normalized_age_bucket_ontology_term_id"]
    )
)

# Map normalized_age_bucket for summary (without modifying meta_data)
mapped_buckets = meta_data["development_stage"].map(stage_to_bucket)

# Count occurrences
counts = mapped_buckets.value_counts().rename_axis("label").reset_index(name="count")

# Convert to list-of-dicts for JSON summary
normalized_age_bucket_ontology_term_id_summary = counts.dropna(subset=["label"]).to_dict(orient="records")
normalized_age_bucket_ontology_term_id_summary

In [ ]:
# Save system prompt to file
with open(system_prompt_file_normalized_age_bucket, "w") as f:
    f.write(system_prompt)

## generate reference data summary json

In [ ]:
def getUniqueLabelCounts (meta_data, column):
    counts = meta_data[column].value_counts().to_dict()
    
    labelCounts = [
        {
            "label": column,
            "cell_count": count,
        }
        for column, count in counts.items()
    ]
    return labelCounts

In [ ]:
existing_summary={}
if os.path.exists(metadata_summary_file):
    with open(metadata_summary_file, "r") as f:
        existing_summary = json.load(f)

summary = {}
# list of manually filled information
summary["reference_name"] = existing_summary.get("reference_name", "")
summary["reference_uuid"] = existing_summary.get("reference_uuid", "")
summary["reference_pyramid"] = existing_summary.get("reference_pyramid", "")
summary["foundation_model"] = existing_summary.get("foundation_model", "")
summary["abstract"] = existing_summary.get("abstract", "")
summary

In [ ]:
# metadata generated from the data
summary["cell_number"] = meta_data.shape[0]

### required metadata used for chart summary
summary["publication"] = publication_summary
summary["tissue"] = getUniqueLabelCounts(meta_data, "tissue")
summary["tissue_type"] = getUniqueLabelCounts(meta_data, "tissue_type")
summary["disease"] = getUniqueLabelCounts(meta_data, "disease")
summary["assay"] = getUniqueLabelCounts(meta_data, "assay")
summary["suspension_type"] = getUniqueLabelCounts(meta_data, "suspension_type")
summary["sex"] = getUniqueLabelCounts(meta_data, "sex")
summary["self_reported_ethnicity"] = getUniqueLabelCounts(meta_data, "self_reported_ethnicity")
summary["normalized_age_bucket"] = normalized_age_bucket_summary
summary["organism"] = getUniqueLabelCounts(meta_data, "normalized_organism")

# other parameters
summary["dataset_ids"] = getUniqueLabelCounts(meta_data, "dataset_id")
summary["tissue_ontology_term_id"] = getUniqueLabelCounts(meta_data, "tissue_ontology_term_id")
summary["disease_ontology_term_id"] = getUniqueLabelCounts(meta_data, "disease_ontology_term_id")
summary["assay_ontology_term_id"] = getUniqueLabelCounts(meta_data, "assay_ontology_term_id")
summary["development_stage"] = getUniqueLabelCounts(meta_data, "development_stage")
summary["development_stage_ontology_term_id"] = getUniqueLabelCounts(meta_data, "development_stage_ontology_term_id")
summary["tissue_type"] = getUniqueLabelCounts(meta_data, "tissue_type")
summary["sex_ontology_term_id"] = getUniqueLabelCounts(meta_data, "sex_ontology_term_id")
summary["self_reported_ethnicity_ontology_term_id"] = getUniqueLabelCounts(meta_data, "self_reported_ethnicity_ontology_term_id")
summary["organism_ontology_term_id"] = getUniqueLabelCounts(meta_data, "organism_ontology_term_id")
summary["normalized_age_bucket_ontology_term_id"] = normalized_age_bucket_ontology_term_id_summary

# cell_type_ontology_term_id
if "custom_cell_type_ontology_term_id" in meta_data:
    summary["cell_type_ontology_term_id"] = getUniqueLabelCounts(meta_data, "custom_cell_type_ontology_term_id")
else:
    summary["cell_type_ontology_term_id"] = getUniqueLabelCounts(meta_data, "cell_type_ontology_term_id")
    
# cell_type
if "custom_cell_type" in meta_data:
    summary["cell_type"] = getUniqueLabelCounts(meta_data, "custom_cell_type")
    note = "'custom_cell_type' is used to generate the cell_type distribution" 
    fwrangle.write(f"{note}\n")
    fwrangle.flush()
    print(note)
else:
    summary["cell_type"] = getUniqueLabelCounts(meta_data, "cell_type")

# output .json file
json_str = to_json_safe(summary, indent=2)
with open(metadata_summary_file, "w") as f:
    f.write(json_str)

## Close wrangling notes

In [ ]:
fwrangle.close()

## Output obs_annotated.tsv.gz

In [ ]:
meta_data.to_csv(metadata_annotated_file, sep="\t", index=False)

## End of standard metadata processing